# Nestlé WISER DOM experiment laboratory

Run top-to-bottom. Edit only the global configuration cell. The notebook preserves the complete experiment workflow, verified checkpoints, aggregate tables, plots, GPU benchmark, and optional IBM hardware study.

**Terminology:** QAOA is the quantum proposal algorithm. `ibm_marrakesh` is a QPU execution backend, not a solver. SciPy/HiGHS and Gurobi are classical MILP backends.


In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path

def _project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/domopt").is_dir():
            return candidate
    raise RuntimeError("Open this notebook from inside wiser-dom-optimization")

PROJECT_ROOT = _project_root(Path.cwd())
src = str(PROJECT_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)
required = ("numpy","pandas","scipy","yaml","matplotlib","nbformat","qiskit","qiskit_ibm_runtime")
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable,"-m","pip","install","--disable-pip-version-check","-e",f"{PROJECT_ROOT}[notebook,ibm]"],check=True)
print(f"Notebook environment ready: {PROJECT_ROOT}")


In [ ]:
import os
from dataclasses import asdict, replace
from pathlib import Path
os.environ["OMP_NUM_THREADS"]="16"
os.environ["OPENBLAS_NUM_THREADS"]="16"
os.environ["MKL_NUM_THREADS"]="16"
os.environ["NUMEXPR_NUM_THREADS"]="16"
import pandas as pd
from IPython.display import Image, Markdown, display
from domopt.checkpoints import StaleCheckpointError, challenge_results_root, checkpoint_identity, checkpoint_run_directory, load_checkpoint, write_checkpoint
from domopt.classical import ClassicalSolverError, available_milp_backends, solve_classical
from domopt.experiments import experiment_profile, ibm_hardware_study_logical_qubits, make_ibm_hardware_study_problem, rank_ibm_hardware_strategies, run_challenge_experiments, run_ibm_hardware_study, write_experiment_results
from domopt.hardware import benchmark_qubo_batch_scoring as _benchmark_qubo_batch_scoring, discover_ibm_backends, hardware_capabilities
from domopt.metrics import compute_metrics
from domopt.poc import POC_REFERENCE_FILENAMES, PocConfig, audit_poc_bundle, audit_poc_outputs, load_poc_problem, prune_pareto_candidates, select_shortage_subset
from domopt.visualization import plot_challenge_results, plot_hardware_benchmark, plot_ibm_backend_snapshot, plot_ibm_hardware_study

PROJECT_ROOT = _project_root(Path.cwd())
BUNDLE_DIR = (PROJECT_ROOT / "data/raw/nestle_challenge").resolve()
PROFILE = "full"
FORCE_RERUN = False
POC_SETTINGS = PocConfig(protection_days=5,min_divert_improvement_fraction=0.05,min_divert_improvement_cases=100,candidate_dc_scope="network_intersection",pareto_prune=False)

PROFILE_OVERRIDES = {
 "sizes":(8,20,50,100,150,200,250,300,372),
 "synthetic_sizes":(50,100,250,500,1_000,2_000,5_000,10_000,20_000,50_000,100_000),
 "scaling_repetitions":3,"base_orders":100,"exact_max_orders":100,"hybrid_max_orders":20_000,"lns_max_orders":372,
 "inventory_shocks":(0.00,0.10,0.25,0.40,0.50,0.55,0.60,0.65,0.70),
 "seeds":(3,11,29,47,71,101,131,173),
 "noise_levels":(0.000,0.010,0.030,0.050,0.075,0.100,0.150,0.200),
 "readout_noise_levels":(0.000,0.005,0.010,0.020,0.030,0.050,0.075,0.100),
 "penalty_scales":(0.25,0.50,1.00,2.00,4.00,8.00),
 "candidate_counts":(1,2,3,4,5,6),
 "qubo_one_hot_multipliers":(1.05,1.15,1.25,1.50,2.00),
 "qubo_pair_multipliers":(0.0,0.5,1.0,1.5,2.0),
}
HYBRID_OVERRIDES = {"iterations":10,"neighborhood_orders":8,"max_qubo_variables":40,"max_candidates_per_order":5,"num_reads":256,"sweeps":300,"top_k_recourse":8,"recourse_time_limit_seconds":20.0,"one_hot_penalty_multiplier":1.25,"pair_penalty_multiplier":1.0,"batch_strategy":"conflict","polish_initial_incumbent":True,"qaoa_layers":1,"qaoa_restarts":16,"qaoa_mixer_topology":"path","milp_backend":"scipy-highs","thread_count":None,"seed":11}
EXACT_LNS_OVERRIDES = {"iterations":16,"minimum_neighborhood_groups":4,"initial_neighborhood_groups":12,"maximum_neighborhood_groups":32,"maximum_neighborhood_orders":120,"maximum_local_fulfillment_variables":25_000,"local_time_limit_seconds":30.0,"mip_relative_gap":0.001,"diversification_interval":3,"adaptive":True,"polish_initial_incumbent":True,"milp_backend":"scipy-highs","thread_count":None,"seed":11}
ENABLED_EXPERIMENTS = {name:True for name in ("solver_comparison","size_scaling","synthetic_scaling","candidate_dc_scope_sensitivity","penalty_weight_sensitivity","qubo_penalty_sensitivity","candidate_count_sensitivity","inventory_shock","qubo_coefficient_noise","qaoa_readout_noise","pareto_pruning_ablation","batch_strategy_ablation","sampler_ablation","synthetic_coordination_control")}
ENABLE_GPU_BENCHMARK = True
ENABLE_IBM_HARDWARE = True
IBM_HARDWARE_PROFILE = "presentation"
IBM_SHOTS = 8_192
IBM_BACKEND_NAME = "ibm_marrakesh"  # QPU execution backend, not a solver
OUTPUT_ROOT = challenge_results_root(PROJECT_ROOT,producer="notebook")

def benchmark_qubo_batch_scoring(*,include_gpu=True,**kwargs):
    return _benchmark_qubo_batch_scoring(variable_counts=(16,32,40,64,96,128,256,512,1_024),sample_counts=(256,4_096,16_384,65_536,131_072),repeats=7,seed=7,include_gpu=include_gpu,**kwargs)

print({"profile":PROFILE,"real_max_groups":372,"synthetic_max_orders":100_000,"scaling_repetitions":3,"common_comparison_groups":100,"hybrid_global_limit":20_000,"local_qubo_cap":40,"hybrid_iterations":10,"exact_lns_iterations":16,"gpu_benchmark":True,"ibm_hardware_enabled":True,"ibm_shots":8_192,"ibm_qpu_backend":IBM_BACKEND_NAME,"output_root":str(OUTPUT_ROOT)})


## 1. Runtime-input readability gate

**Purpose.** Verify that the five canonical runtime tables are present, readable, and non-empty.

**Why it matters.** A missing or misidentified upload invalidates every downstream result.


In [ ]:
file_audit = audit_poc_bundle(BUNDLE_DIR)
assert file_audit["readable"].all()
display(file_audit[["role","filename","rows","columns","readable"]])


## 2. Build and audit the real POC model

**Purpose.** Construct the unpruned and Pareto-pruned challenge models and audit optional reference outputs.

**Why it matters.** This checks grain, keys, candidates, resource accounting, and source consistency before optimization.


In [ ]:
problem_unpruned = load_poc_problem(BUNDLE_DIR,config=POC_SETTINGS,strict_bundle_audit=False)
problem_pruned = prune_pareto_candidates(problem_unpruned)
display(pd.DataFrame([
 {"variant":"unpruned","orders":len(problem_unpruned.orders),"assignment_groups":problem_unpruned.orders["assignment_group"].nunique(),"order_lines":len(problem_unpruned.order_lines),"candidate_rows":len(problem_unpruned.candidates)},
 {"variant":"pareto_pruned","orders":len(problem_pruned.orders),"assignment_groups":problem_pruned.orders["assignment_group"].nunique(),"order_lines":len(problem_pruned.order_lines),"candidate_rows":len(problem_pruned.candidates)},
]))
reference_available = all((BUNDLE_DIR/name).is_file() for name in POC_REFERENCE_FILENAMES.values())
if reference_available:
    display(pd.Series(audit_poc_outputs(BUNDLE_DIR,problem_unpruned),name="reference audit"))
else:
    print("Optional recommendation outputs are absent; reconciliation is skipped.")


## 3. Verified experiment runner

**Purpose.** Run or resume every enabled study under one verified configuration.

**Why it matters.** Checkpoints validate configuration, source, schema, row count, and content hash. Only the frozen-routing control may be infeasible under a severe inventory shock; every other infeasible row stops the notebook.


In [ ]:
experiment_frames = {}
BASE_PROFILE = experiment_profile(PROFILE)
PROFILE_SETTINGS = replace(BASE_PROFILE,**PROFILE_OVERRIDES,hybrid=replace(BASE_PROFILE.hybrid,**HYBRID_OVERRIDES),exact_lns=replace(BASE_PROFILE.exact_lns,**EXACT_LNS_OVERRIDES))
PROFILE_CONFIGURATION = {"experiment_profile":asdict(PROFILE_SETTINGS),"poc":asdict(POC_SETTINGS)}
SUITE_IDENTITY = checkpoint_identity(problem_unpruned,profile=PROFILE,experiment="challenge_suite",configuration=PROFILE_CONFIGURATION)
OUTPUT_DIR = checkpoint_run_directory(OUTPUT_ROOT,SUITE_IDENTITY)
TABLE_DIR, FIGURE_DIR = OUTPUT_DIR/"tables", OUTPUT_DIR/"figures"
TABLE_DIR.mkdir(parents=True,exist_ok=True); FIGURE_DIR.mkdir(parents=True,exist_ok=True)

def feasible_mask(frame):
    values=frame["feasible"]
    return values if values.dtype==bool else values.astype(str).str.lower().isin({"true","1"})

def run_or_load(name):
    if not ENABLED_EXPERIMENTS.get(name,False): return None
    identity=checkpoint_identity(problem_unpruned,profile=PROFILE,experiment=name,configuration=PROFILE_CONFIGURATION)
    path=TABLE_DIR/f"{name}.csv"; frame=None
    if not FORCE_RERUN:
        try:
            frame=load_checkpoint(path,identity); print(f"loaded verified checkpoint: {path.relative_to(PROJECT_ROOT)}")
        except StaleCheckpointError as error:
            print(f"checkpoint unavailable or stale ({error}); recomputing {name}")
    if frame is None:
        frame=run_challenge_experiments(problem_unpruned,profile=PROFILE_SETTINGS,experiments=[name])
        write_experiment_results(frame,path); write_checkpoint(frame,path,identity)
        print(f"wrote verified checkpoint: {path.relative_to(PROJECT_ROOT)}")
    invalid=~feasible_mask(frame)
    if invalid.any():
        error_type=frame["error_type"] if "error_type" in frame else pd.Series("",index=frame.index)
        error_message=frame["error_message"] if "error_message" in frame else pd.Series("",index=frame.index)
        expected=(frame["experiment"].eq("inventory_shock") & frame["level"].str.contains("fixed_routing_recourse",na=False) & error_type.eq("ClassicalSolverError") & error_message.str.contains(r"status=2|infeasible",case=False,na=False,regex=True))
        unexpected=frame.loc[invalid & ~expected]
        if not unexpected.empty:
            cols=[c for c in ("experiment","level","validation_categories","validation_violation_count","error_type","error_message") if c in unexpected]
            raise RuntimeError(f"Unexpected infeasible experiment rows: {unexpected[cols].to_dict('records')}")
        if expected.any(): print("Expected stress-test infeasibility:",frame.loc[expected,[c for c in ("level","error_type","error_message") if c in frame]].to_dict("records"))
    experiment_frames[name]=frame
    return frame

VIEWS={
"solver_comparison":(["method","feasible","objective_value","requested_value","objective_capture_rate","case_fill_rate","reassigned_orders","penalty_cost","shipping_cost","runtime_seconds","optimality_gap","initial_polish_improvement","search_improvement","maximum_local_variables","maximum_qubo_variables"],"objective_value",False),
"size_scaling":(["method","actual_assignment_groups","repetition","order_count","order_line_count","candidate_count","maximum_local_variables","maximum_qubo_variables","objective_value","objective_capture_rate","case_fill_rate","runtime_seconds","feasible"],["actual_assignment_groups","method","repetition"],True),
"synthetic_scaling":(["method","actual_assignment_groups","repetition","generator_seed","candidate_count","maximum_local_variables","maximum_qubo_variables","objective_capture_rate","runtime_seconds","feasible"],["actual_assignment_groups","method","repetition"],True),
"candidate_dc_scope_sensitivity":(["candidate_dc_scope","method","candidate_count","objective_capture_rate","case_fill_rate","runtime_seconds","feasible"],["candidate_dc_scope","method"],True),
"penalty_weight_sensitivity":(["penalty_scale","method","case_fill_rate","reassigned_orders","penalty_cost","shipping_cost","runtime_seconds"],["penalty_scale","method"],True),
"qubo_penalty_sensitivity":(["one_hot_penalty_multiplier","pair_penalty_multiplier","raw_one_hot_rate","hybrid_improvement","accepted_moves","recourse_solves","runtime_seconds"],["one_hot_penalty_multiplier","pair_penalty_multiplier"],True),
"candidate_count_sensitivity":(["candidate_limit","method","candidate_count","maximum_qubo_variables","objective_value","case_fill_rate","runtime_seconds"],["candidate_limit","method"],True),
"inventory_shock":(["inventory_shock","method","feasible","objective_value","objective_capture_rate","case_fill_rate","unassigned_orders","penalty_cost","runtime_seconds"],["inventory_shock","method"],True),
"qubo_coefficient_noise":(["seed","coefficient_noise_relative_sigma","raw_one_hot_rate","hybrid_improvement","accepted_moves","runtime_seconds"],["coefficient_noise_relative_sigma","seed"],True),
"qaoa_readout_noise":(["seed","qaoa_readout_bitflip_probability","raw_one_hot_rate","hybrid_improvement","accepted_moves","runtime_seconds","feasible"],["qaoa_readout_bitflip_probability","seed"],True),
"pareto_pruning_ablation":(["level","candidate_count","maximum_qubo_variables","hybrid_improvement","runtime_seconds","feasible"],None,True),
"batch_strategy_ablation":(["level","hybrid_improvement","accepted_moves","maximum_qubo_variables","recourse_solves","runtime_seconds"],None,True),
"sampler_ablation":(["level","raw_one_hot_rate","initial_polish_improvement","hybrid_improvement","accepted_moves","quantum_simulator_calls","runtime_seconds","feasible"],None,True),
"synthetic_coordination_control":(["method","objective_value","case_fill_rate","runtime_seconds","optimality_gap","raw_initial_objective","initial_polish_improvement","hybrid_improvement","total_hybrid_improvement","feasible"],"objective_value",False),
}

def show_experiment(name):
    frame=run_or_load(name)
    if name=="inventory_shock":
        frame=frame.copy(); mask=frame["level"].str.contains("fixed_routing_recourse",na=False)&~feasible_mask(frame); frame.loc[mask,"method"]="fixed_routing_recourse (infeasible)"
    cols,sort_by,ascending=VIEWS[name]; view=frame[[c for c in cols if c in frame]]
    if sort_by is not None: view=view.sort_values(sort_by,ascending=ascending)
    display(view)
print(f"artifact scope: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")


## 4. Common solver comparison

**Purpose.** Compare default, greedy, polished greedy, exact LNS, full MILP, and hybrid under one evaluator.

**Why it matters.** This establishes the production hierarchy.


In [ ]:
show_experiment("solver_comparison")


## 5. Optional MILP-backend comparison

**Purpose.** Compare SciPy/HiGHS and licensed Gurobi on the same compiled MILP.

**Why it matters.** These are classical MILP backends. With a time limit or nonzero MIP gap, different valid incumbents are normal; equality is asserted only when all completed backends prove optimality.


In [ ]:
backend_problem=select_shortage_subset(problem_unpruned,PROFILE_SETTINGS.base_orders)
rows=[]
for backend in ("scipy-highs","gurobi"):
    if backend=="gurobi" and not available_milp_backends()["gurobi"]:
        rows.append({"milp_backend":backend,"status":"skipped: gurobipy is not installed","feasible":None}); continue
    try:
        solution=solve_classical(backend_problem,backend=backend,time_limit_seconds=60,mip_relative_gap=0.01,seed=PROFILE_SETTINGS.exact_lns.seed)
        metrics=compute_metrics(backend_problem,solution)
        rows.append({"milp_backend":backend,"status":"completed","feasible":metrics["feasible"],"objective_value":metrics["objective_value"],"objective_capture_rate":metrics["objective_capture_rate"],"case_fill_rate":metrics["case_fill_rate"],"runtime_seconds":metrics["runtime_seconds"],"optimality_gap":metrics["optimality_gap"]})
    except ClassicalSolverError as error:
        rows.append({"milp_backend":backend,"status":f"skipped: {error}","feasible":None})
backend_comparison=pd.DataFrame(rows)
completed=backend_comparison.loc[backend_comparison["status"].eq("completed")].copy()
assert completed["feasible"].all()
backend_comparison["objective_gap_to_best"]=pd.NA; backend_comparison["objective_gap_to_best_pct"]=pd.NA
if len(completed)>=2:
    proved=completed["optimality_gap"].fillna(float("inf")).abs().le(1e-9)
    if proved.all():
        spread=completed["objective_value"].max()-completed["objective_value"].min()
        assert abs(spread)<=1e-6*max(1.0,completed["objective_value"].abs().max())
    else:
        best=completed["objective_value"].max(); gaps=best-completed["objective_value"]
        backend_comparison.loc[completed.index,"objective_gap_to_best"]=gaps
        backend_comparison.loc[completed.index,"objective_gap_to_best_pct"]=gaps/max(1.0,abs(best))*100
backend_comparison.to_csv(TABLE_DIR/"milp_backend_comparison.csv",index=False)
display(backend_comparison)


## 6. Real assignment-group size scaling

**Purpose.** Scale all applicable methods across nested real assignment-group subsets.

**Why it matters.** Assignment groups are the actual atomic routing decisions.


In [ ]:
show_experiment("size_scaling")


## 7. Controlled synthetic scaling

**Purpose.** Repeat independent synthetic instances through 100,000 orders.

**Why it matters.** This separates complexity from real-subset composition.


In [ ]:
show_experiment("synthetic_scaling")


## 8. Candidate-DC universe sensitivity

**Purpose.** Compare candidate-universe definitions.

**Why it matters.** Candidate generation can silently cap quality.


In [ ]:
show_experiment("candidate_dc_scope_sensitivity")


## 9. Business penalty-weight sensitivity

**Purpose.** Scale unmet-demand penalties and compare operational trade-offs.

**Why it matters.** This tests whether recommendations depend on one coefficient.


In [ ]:
show_experiment("penalty_weight_sensitivity")


## 10. QUBO penalty calibration

**Purpose.** Sweep one-hot and pairwise encoding penalties.

**Why it matters.** Encoding choices must not arbitrarily dominate proposal quality.


In [ ]:
show_experiment("qubo_penalty_sensitivity")


## 11. Candidate-count sensitivity

**Purpose.** Vary retained alternatives per assignment group.

**Why it matters.** Candidate caps control width and runtime.


In [ ]:
show_experiment("candidate_count_sensitivity")


## 12. Inventory-shock robustness

**Purpose.** Compare fixed nominal routing with adaptive reoptimization.

**Why it matters.** A fixed policy may become mathematically infeasible while adaptive routing remains valid.


In [ ]:
show_experiment("inventory_shock")


## 13. Seed and QUBO coefficient-noise robustness

**Purpose.** Perturb local QUBO coefficients across eight seeds.

**Why it matters.** This identifies proposal-quality transition regions.


In [ ]:
show_experiment("qubo_coefficient_noise")


## 14. Local QAOA readout-noise proxy

**Purpose.** Apply measurement bit flips before one-hot repair.

**Why it matters.** This isolates readout sensitivity from gate/decoherence effects.


In [ ]:
show_experiment("qaoa_readout_noise")


## 15. Heuristic Pareto-pruning ablation

**Purpose.** Compare pruned and unpruned candidate sets.

**Why it matters.** Shared resources mean isolated dominance is not globally lossless.


In [ ]:
show_experiment("pareto_pruning_ablation")


## 16. Random versus conflict-based batches

**Purpose.** Compare random and resource-conflict neighborhoods.

**Why it matters.** Problem-aware decomposition should focus interacting decisions.


In [ ]:
show_experiment("batch_strategy_ablation")


## 17. Sampler and quantum-simulation ablation

**Purpose.** Compare exact feasible, random, annealing, and QAOA proposals.

**Why it matters.** The downstream solver is held fixed to isolate proposal quality.


In [ ]:
show_experiment("sampler_ablation")


## 18. Synthetic coordination control

**Purpose.** Expose a verified greedy trap and compare coordinated methods.

**Why it matters.** This demonstrates representational capability, not quantum advantage.


In [ ]:
show_experiment("synthetic_coordination_control")


## 19. Persist aggregate evidence and graphics

**Purpose.** Rebuild one aggregate from every enabled verified checkpoint and generate all figures.

**Why it matters.** This prevents an incomplete aggregate after rerunning the helper cell interactively.


In [ ]:
frames=[]
for name,enabled in ENABLED_EXPERIMENTS.items():
    if not enabled: continue
    frame=experiment_frames.get(name)
    if frame is None: frame=run_or_load(name)
    if frame is not None: frames.append(frame.dropna(axis=1,how="all"))
if frames:
    results=pd.concat(frames,ignore_index=True,sort=False)
    identity=checkpoint_identity(problem_unpruned,profile=PROFILE,experiment="aggregate_results",configuration=PROFILE_CONFIGURATION)
    path=write_experiment_results(results,OUTPUT_DIR/"aggregate_results.csv"); write_checkpoint(results,path,identity)
    figure_paths=plot_challenge_results(results,FIGURE_DIR)
    print(f"wrote {len(results)} aggregate rows from {len(frames)} experiment families")
    for name,path in figure_paths.items(): display(Markdown(f"### {name.replace('_',' ').title()}")); display(Image(filename=str(path)))
else: print("No enabled experiment frames.")


## 20. Optional GPU crossover benchmark

**Purpose.** Measure batched QUBO scoring on CPU and GPU.

**Why it matters.** This is a scoring crossover, not an end-to-end GPU DOM-solver claim.


In [ ]:
capabilities=hardware_capabilities(); display(pd.Series(capabilities,name="hardware capability"))
if ENABLE_GPU_BENCHMARK:
    hardware_results=benchmark_qubo_batch_scoring(include_gpu=True)
    hardware_results.to_csv(OUTPUT_DIR/"hardware_qubo_scoring.csv",index=False)
    figure=plot_hardware_benchmark(hardware_results,FIGURE_DIR/"hardware_qubo_scoring.png")
    display(hardware_results); display(Image(filename=str(figure)))
else: print("GPU benchmark skipped.")


## 21. IBM QPU-backend discovery and QAOA hardware stress test

**Purpose.** Run the 18-job, 8,192-shot QAOA matrix on one IBM QPU backend.

**Why it matters.** The IBM device executes QAOA circuits; it is not a solver. Exact recourse and validation retain final feasibility, and no Nestlé identifiers are sent.


In [ ]:
if ENABLE_IBM_HARDWARE:
    logical_qubits=ibm_hardware_study_logical_qubits(); snapshot=discover_ibm_backends(min_num_qubits=logical_qubits)
    qpu_backend=IBM_BACKEND_NAME or str(snapshot.loc[snapshot["selected_least_busy"],"backend"].iloc[0])
    if qpu_backend not in set(snapshot["backend"].astype(str)): raise RuntimeError(f"Requested QPU backend {qpu_backend!r} is not eligible")
    snapshot["selected_for_study"]=snapshot["backend"].astype(str).eq(qpu_backend)
    problem=make_ibm_hardware_study_problem()
    identity=checkpoint_identity(problem,profile=f"ibm-{IBM_HARDWARE_PROFILE}",experiment="ibm_hardware_stress",configuration={"qpu_backend":qpu_backend,"shots":IBM_SHOTS,"hardware_profile":IBM_HARDWARE_PROFILE,"logical_qubits":logical_qubits,"qaoa_matrix":"p=1,2 x baseline,dd,dd+measurement-twirling","data_scope":"independently generated synthetic control"})
    root=checkpoint_run_directory(OUTPUT_ROOT,identity); tables,figures=root/"tables",root/"figures"; tables.mkdir(parents=True,exist_ok=True); figures.mkdir(parents=True,exist_ok=True)
    snapshot.to_csv(root/"ibm_qpu_backend_snapshot.csv",index=False); display(snapshot); display(Image(filename=str(plot_ibm_backend_snapshot(snapshot,figures/"ibm_qpu_backend_queue.png"))))
    target=tables/"ibm_hardware_stress.csv"; existing=None
    if not FORCE_RERUN:
        try: existing=load_checkpoint(target,identity); print(f"resuming from {len(existing)} verified rows")
        except StaleCheckpointError as error: print(f"no reusable IBM checkpoint ({error})")
    qpu_results=run_ibm_hardware_study(allow_remote=True,backend_name=qpu_backend,shots=IBM_SHOTS,profile=IBM_HARDWARE_PROFILE,progress_callback=lambda frame:write_checkpoint(frame,target,identity),existing_results=existing)
    qpu_path,_=write_checkpoint(qpu_results,target,identity); ranking=rank_ibm_hardware_strategies(qpu_results); ranking_path=write_experiment_results(ranking,tables/"ibm_qaoa_configuration_ranking.csv")
    cols=[c for c in ("level","hardware_backend","hardware_mitigation_strategy","qaoa_layers","feasible","error_type","error_message","search_improvement","raw_one_hot_rate","hardware_qubo_optimal_hit_rate","hardware_transpiled_depth","hardware_two_qubit_gates","hardware_queue_seconds","hardware_execution_seconds","hardware_quantum_seconds","hardware_returned_samples","hardware_feasible_shots","runtime_seconds") if c in qpu_results]
    display(qpu_results[cols]); display(ranking)
    if bool(ranking["selected_best_observed"].any()):
        best=ranking.loc[ranking["selected_best_observed"]].iloc[0]; display(Markdown(f"**Best observed QAOA configuration on `{qpu_backend}`:** {best['variant']} (median exact feasible-QUBO raw hit rate {100*best['hardware_qubo_optimal_hit_rate']:.1f}%).")); display(Image(filename=str(plot_ibm_hardware_study(qpu_results,figures/"ibm_qaoa_hardware_stress.png"))))
    print(f"wrote {qpu_path.relative_to(PROJECT_ROOT)}"); print(f"wrote {ranking_path.relative_to(PROJECT_ROOT)}")
else: print("IBM QPU study skipped.")


## 22. Interpretation guardrails

Every returned plan must pass the independent validator. Real POC evidence remains separate from synthetic controls. The QPU experiment evaluates QAOA executed on an IBM backend; it does not establish quantum advantage or treat the backend as a solver. Do not add a generic risk coefficient without calibrated forecast scenarios and probabilities.
